In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.silver;

CREATE OR REPLACE TEMP VIEW orders_typed AS
SELECT
  order_id,
  customer_id,
  order_status,
  TRY_CAST(order_purchase_timestamp      AS TIMESTAMP) AS order_purchase_timestamp,
  TRY_CAST(order_approved_at             AS TIMESTAMP) AS order_approved_at,
  TRY_CAST(order_delivered_carrier_date  AS TIMESTAMP) AS order_delivered_carrier_date,
  TRY_CAST(order_delivered_customer_date AS TIMESTAMP) AS order_delivered_customer_date,
  TRY_CAST(order_estimated_delivery_date AS DATE)      AS order_estimated_delivery_date,
  _ingested_at
FROM workspace.bronze.orders
QUALIFY ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY _ingested_at DESC) = 1;

CREATE OR REPLACE TABLE workspace.silver.orders AS
SELECT * FROM orders_typed
WHERE order_id IS NOT NULL AND customer_id IS NOT NULL AND order_purchase_timestamp IS NOT NULL;

CREATE OR REPLACE TABLE workspace.silver.orders_quarantine AS
SELECT *, 'llave o fecha de compra nula' AS motivo
FROM orders_typed
WHERE order_id IS NULL OR customer_id IS NULL OR order_purchase_timestamp IS NULL;